<h1 class='text-center'>⭐EVER LEARNERS⭐</h1>

## LLM (QWEN3VL)

### Transfer to Kaggle Working Area if you are using Kaggle. If not Just get the module at the right place

In [69]:
## Only Kaggle, For Colab and Locally not needed at all!
!cp -r /kaggle/input/barbados-hugging-face-data-extraction/data_extraction  /kaggle/working/
!cp -r /kaggle/input/barbados-llm-training-pipeline/llm_training_pipeline /kaggle/working/
!cp -r /kaggle/input/yolo-polygon-pipeline/polygon_training_inference /kaggle/working/

### Below is the full pipeline, If you want to train and infer!

In [ ]:
# If you are having indent errors, please use an LLM to fix that, it is not really a big deal🤣

# !python /kaggle/working/data_extraction/main_data_pipeline.py\
#   --data_path /kaggle/input/barbados-survey-plans/dataset/survey_plans/\
#   --train_csv_path /kaggle/input/barbados-survey-plans/dataset/Train.csv\
#   --test_csv_path /kaggle/input/barbados-survey-plans/dataset/Test.csv\
#   --ss_csv_path /kaggle/input/barbados-survey-plans/dataset/SampleSubmission.csv

# !python /kaggle/working/llm_training_pipeline/train_pipeline.py

# !python /kaggle/working/llm_training_pipeline/inference.py\ --temperature 0 \ --max_new_tokens 512

### 🔍 Real-World Inference Example

The script below demonstrates how to use the trained model for real-world inference on a new test set.

It takes the path of an input image, processes it through the model, and outputs the corresponding predictions or visual understanding.

This step simulates how the model performs when exposed to unseen, real-world data — ensuring that it generalizes beyond the training examples.

In [19]:
import os
import shutil
import pandas as pd
import random
import re

def create_test_csv_from_anonymised_images(
    source_dir: str,
    output_dir: str,
    output_csv_path: str,
    sample_count: int = 5
):
    """
    Copies anonymised images and creates a CSV with extracted IDs.

    Args:
        source_dir (str): Directory with anonymised_*.jpg images
        output_dir (str): Directory to copy selected images
        output_csv_path (str): Path to save Test.csv
        sample_count (int): Number of images to copy
    """
    os.makedirs(output_dir, exist_ok=True)

    # Find all anonymised images
    image_files = [
        f for f in os.listdir(source_dir)
        if f.lower().startswith("anonymised_") and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    if not image_files:
        raise FileNotFoundError(f"No anonymised images found in {source_dir}")

    # Sample subset
    sampled = random.sample(image_files, min(sample_count, len(image_files)))

    ids = []
    for fname in sampled:
        # Extract ID from filename, e.g. anonymised_4512-063.jpg → 4512-063
        match = re.search(r"anonymised_(.+?)\.", fname)
        if match:
            ID = match.group(1)
            ids.append({"ID": ID})

            shutil.copy(
                os.path.join(source_dir, fname),
                os.path.join(output_dir, fname)
            )

    # Save CSV with IDs only
    test_df = pd.DataFrame(ids)
    test_df.to_csv(output_csv_path, index=False)

    print(f"✅ Copied {len(ids)} anonymised image(s) → {output_dir}")
    print(f"✅ Created Test CSV → {output_csv_path}")
    print(test_df.head())

    return test_df


# Example usage
test_df = create_test_csv_from_anonymised_images(
    source_dir="/kaggle/input/barbados-survey-plans/dataset/survey_plans/",
    output_dir="./inference_samples/",
    output_csv_path="./Test.csv",
    sample_count=1
)

✅ Copied 1 anonymised image(s) → ./inference_samples/
✅ Created Test CSV → ./Test.csv
         ID
0  7704-122


In [ ]:
!python /kaggle/working/data_extraction/test_data_pipeline.py \
  --data_path /kaggle/working/inference_samples
!python /kaggle/working/llm_training_pipeline/inference.py \
  --model_path CodeJoe/barbados-Qwen3-mapper \
  --temperature 0 \
  --max_new_tokens 512

## 🚀 Running Inference Pipeline

The commands below demonstrate how to **run the full inference workflow** — starting from test data preparation to model inference using the fine-tuned Qwen3 model.

### 🧩 Step 1: Prepare the Inference Data

This command executes the **data extraction script**, which loads and formats the new images for inference.
Make sure your sample images are located in the `/kaggle/working/inference_samples` directory.

```bash
!python /kaggle/working/data_extraction/test_data_pipeline.py \
  --data_path /kaggle/working/inference_samples
```

---

### 🤖 Step 2: Run the Model Inference

Once the data is ready, this command uses the trained model
**`CodeJoe/barbados-Qwen3-mapper`** to generate predictions.
It performs inference with deterministic settings (`temperature=0`)
and limits the generated output to 512 tokens.

```bash
!python /kaggle/working/llm_training_pipeline/inference.py \
  --model_path CodeJoe/barbados-Qwen3-mapper \
  --temperature 0 \
  --max_new_tokens 512
```

---

✅ **Summary:**
These two commands:

1. **Prepare the new dataset** for evaluation.
2. **Run inference** using your trained Qwen3-based vision-language model.

This workflow enables **real-world evaluation** of your model on unseen data, ensuring that it performs well beyond the training samples.

In [53]:
!python /kaggle/working/data_extraction/main_data_pipeline.py \
  --data_path /kaggle/working/inference_samples
!python /kaggle/working/llm_training_pipeline/inference.py \
  --model_path CodeJoe/barbados-Qwen3-mapper \
  --temperature 0 \
  --max_new_tokens 512

──────────────────────────── 🤗 Inference Dataset Preparation Pipeline ─────────────────────────────

🖼️ Copying All Test Images
────────────────────────────────────────────────────────────────────────────────────────────────────
  Copying images... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:0000:00
✅ Copied 3 images to: ./inference_images/

📋 Creating Test CSV
────────────────────────────────────────────────────────────────────────────────────────────────────
✅ Test CSV created at: ./inference_images/Test.csv
 Generated  
  Test CSV  
  Preview   
╭──────────╮
│ ID       │
├──────────┤
│ 7711-040 │
│ BYLS-039 │
│ 7704-122 │
╰──────────╯

🤗 Creating HuggingFace Test Dataset
────────────────────────────────────────────────────────────────────────────────────────────────────
📦 Converting to HuggingFace Dataset format...
🖼️  Casting image column...
✅ HuggingFace Dataset created: 3 examples

💾 Validating and Saving Dataset
─────────────────────────────────────────────────────────────────

## YOLOV8M

In [60]:
!pip install -U ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.3 MB/s eta 0:00:00:00:0100:01



## 🚀 Inference Examples

### 🧪 1. Run Inference on Official Test Set

Use this to generate predictions for the provided **Barbados Survey Plans test dataset**.

```bash
!python /kaggle/working/polygon_training_inference/run_pipeline.py inference \
  --test_csv /kaggle/input/barbados-survey-plans/dataset/Test.csv \
  --image_dir /kaggle/input/barbados-survey-plans/dataset/survey_plans \
  --output_path /kaggle/working/submission.csv \
  --use_pretrained \
  --format_type list
```

This command:

* Uses the pretrained model from **Hugging Face**
* Runs inference on the official test images
* Saves predictions to `submission.csv` in **list format** (`[(x,y), ...]`)

---

### 🌍 2. Run Inference on Any Unseen Data

Use this when you want to test the model on **your own custom images** (new real-world data).

```bash
!python /kaggle/working/polygon_training_inference/run_pipeline.py inference \
  --test_csv /kaggle/working/inference_images/Test.csv \
  --image_dir /kaggle/working/inference_images \
  --output_path /kaggle/working/submission.csv \
  --use_pretrained \
  --format_type list
```

This command:

* Loads the same pretrained model from Hugging Face
* Runs inference on any new images placed in `/kaggle/working/inference_images/`
* Saves outputs to `submission.csv` with coordinates in list format

---

✅ **Tip:**
Make sure your `Test.csv` file inside `inference_images/` contains an `ID` column matching the filenames of your test images (e.g. `anonymised_12345.jpg`).


In [80]:
!python /kaggle/working/polygon_training_inference/run_pipeline.py inference \
  --test_csv /kaggle/input/barbados-survey-plans/dataset/Test.csv \
  --image_dir /kaggle/input/barbados-survey-plans/dataset/survey_plans \
  --output_path /kaggle/working/submission.csv \
  --use_pretrained \
  --format_type list

🚀 Running Inference Pipeline...
📥 Downloading model from Hugging Face: CodeJoe/barbados-yolov8m-mapper
✅ Model downloaded to: /root/.cache/huggingface/hub/models--CodeJoe--barbados-yolov8m-mapper/snapshots/aae381d04986faf83bc840f13aa103333d68cfe0/segment/train/weights/best.pt
🚀 Model loaded and ready for inference!

🚀 Generating submission...
100%|█████████████████████████████████████████| 219/219 [00:43<00:00,  5.06it/s]

✅ Submission saved to: /kaggle/working/submission.csv
   Total predictions: 219
   Empty predictions: 0
✅ Inference complete! Results saved to: /kaggle/working/submission.csv


In [81]:
pd.read_csv('/kaggle/working/submission.csv')

,ID,geometry
0,7703-078,"[(1146.0, -2801.0), (960.0, -2643.0), (1008.0,..."
1,8606-095,"[(777.0, -2853.0), (482.0, -2191.0), (1707.0, ..."
2,7703-064,"[(1241.0, -2623.0), (789.0, -2191.0), (1119.0,..."
3,7703-101,"[(910.0, -3238.0), (482.0, -2480.0), (1646.0, ..."
4,7707-114,"[(1444.0, -2610.0), (937.0, -2147.0), (1218.0,..."
...,...,...
214,BYLS-083,"[(668.0, -2570.0), (709.0, -2225.0), (1765.0, ..."
215,BYLS-084,"[(1110.0, -2690.0), (910.0, -2407.0), (1478.0,..."
216,BYLS-114,"[(1019.0, -2968.0), (1018.0, -1670.0), (1528.0..."
217,BYLS-115,"[(837.0, -2529.0), (816.0, -2129.0), (1623.0, ..."


In [78]:
!python /kaggle/working/polygon_training_inference/run_pipeline.py inference \
  --test_csv /kaggle/working/inference_images/Test.csv \
  --image_dir /kaggle/working/inference_images \
  --output_path /kaggle/working/submission.csv \
  --use_pretrained \
  --format_type list

🚀 Running Inference Pipeline...
📥 Downloading model from Hugging Face: CodeJoe/barbados-yolov8m-mapper
✅ Model downloaded to: /root/.cache/huggingface/hub/models--CodeJoe--barbados-yolov8m-mapper/snapshots/aae381d04986faf83bc840f13aa103333d68cfe0/segment/train/weights/best.pt
🚀 Model loaded and ready for inference!

🚀 Generating submission...
100%|█████████████████████████████████████████████| 3/3 [00:01<00:00,  1.75it/s]

✅ Submission saved to: /kaggle/working/submission.csv
   Total predictions: 3
   Empty predictions: 0
✅ Inference complete! Results saved to: /kaggle/working/submission.csv


In [79]:
pd.read_csv('/kaggle/working/submission.csv')

,ID,geometry
0,7711-040,"[(1247.0, -2880.0), (802.0, -2389.0), (1424.0,..."
1,BYLS-039,"[(1356.0, -2543.0), (842.0, -2079.0), (1120.0,..."
2,7704-122,"[(932.0, -2843.0), (735.0, -2621.0), (1549.0, ..."


In [82]:
import pandas as pd

# Load both CSV files
vlm_df = pd.read_csv("/kaggle/working/vlm_parsed_clean_postprocessed.csv")
submission_df = pd.read_csv("/kaggle/working/submission.csv")

# Preview columns
print("VLM columns:", vlm_df.columns.tolist())
print("Submission columns:", submission_df.columns.tolist())

# 🧠 Merge both DataFrames on the 'ID' column
merged_df = pd.merge(vlm_df, submission_df, on="ID", how="left")

VLM columns: ['Land Surveyor', 'Surveyed For', 'Certified date', 'Total Area', 'Unit of Measurement', 'Address', 'Parish', 'LT Num', 'ID']
Submission columns: ['ID', 'geometry']


In [ ]:
# Save merged result
merged_df.to_csv("Combined_submission.csv", index=False)

print("✅ Merged file saved to /kaggle/working/merged_output.csv")
print(f"Total rows: {len(merged_df)}")